# GPT Researcher with Gemini 2.5 - Demo Notebook

This notebook demonstrates how to use GPT Researcher with Google Gemini 2.5 models and Gemini Grounding with Google Search.

## Setup

### Installation Options:

**Option 1: Install in development mode (recommended)**
```bash
# From the repository root directory
pip install -e .
```

**Option 2: Run the cell below to add repo to Python path**
- This allows importing without installation
- Only works while notebook is running

### Requirements:
1. Your Gemini API key from https://aistudio.google.com/app/apikey
2. Required packages: `pip install -r requirements.txt` (from repo root)

## 1. Initial Setup and Configuration

In [1]:
import sys
import os
import re
import asyncio 
from pathlib import Path

# Add repo to path
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

# Load environment variables
from gpt_researcher import GPTResearcher 
from dotenv import load_dotenv
load_dotenv()

# Verify settings
print(f"✅ RETRIEVER: {os.environ.get('RETRIEVER')}")
print(f"✅ GEMINI_API_KEY: {'Set' if os.environ.get('GEMINI_API_KEY') else 'Missing'}") 


def clean_filename(text):
    # Replace invalid characters with an underscore or empty string
    return re.sub(r'[\\/*?:"<>|]', "", text).strip()

✅ RETRIEVER: gemini_grounding
✅ GEMINI_API_KEY: Set


## 2. Basic Research Example

Let's run a simple research query about current events.

In [2]:
async def basic_research(query, instructions=None):
    """
    Basic research example using Gemini Grounding
    """
    print("🔍 Starting research...\n")
    
    # Create researcher instance
    researcher = GPTResearcher(
        query=query,
        report_type="deep",
        verbose=True
    )
    
    # Conduct research
    print("📚 Gathering information...")
    research_context = await researcher.conduct_research()
    
    # Generate report
    # 3. Switch the report type to "blog_report" before writing
    researcher.report_type = "blog_report"

    # 4. Generate the report (uses deep context + blog prompt + extra instructions if any)
    report = await researcher.write_report(custom_prompt=instructions) 
    
    # Display results
    print("\n" + "="*80)
    print("📄 RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    # Show sources
    sources = researcher.get_source_urls()
    print("\n" + "="*80)
    print(f"📎 SOURCES ({len(sources)} total)")
    print("="*80)
    for i, source in enumerate(sources, 1):
        print(f"{i}. {source}")
    
    # Show costs
    costs = researcher.get_costs()
    print(f"\n💰 Total cost: ${costs:.4f}")
    
    return report, sources

In [3]:
research_tasks = [
    {
        "query": "Table Extraction from PDFs: Turning Broken Tables into Reliable Structured Data",
        "instructions": """
Focus on the common business and developer pain around extracting tables from PDFs.

Cover:
- Why PDF tables are difficult: merged cells, missing borders, multi-line headers, footnotes, nested rows, and multi-page tables
- Why traditional OCR often flattens tables into unusable text
- How broken table extraction affects finance, logistics, insurance, and legal workflows
- Common failed approaches: copy-paste, rule-based OCR, brittle coordinate templates, and manual cleanup

Clearly explain how DocumentLens helps:
- Uses layout-aware analysis to detect table boundaries, rows, columns, and relationships
- Preserves table structure instead of flattening content
- Extracts tables into structured JSON/CSV/XML outputs
- Grounds table cells back to the original page location for review
- Supports complex scanned PDFs and regional document formats

Position DocumentLens as a practical solution for teams that need reliable table extraction from messy PDFs at enterprise scale.
""",
        "keywords": [
            "table extraction from PDF",
            "PDF table extraction",
            "extract tables from PDF",
            "AI table extraction",
            "structured table extraction",
            "DocumentLens table extraction"
        ]
    },
    {
        "query": "PDF Table Extraction for Developers: From Raw Documents to Clean JSON",
        "instructions": """
Write for a technical audience searching for better PDF table extraction methods.

Cover:
- Why developers struggle with PDF tables using Python libraries, OCR tools, and coordinate-based parsers
- Problems with inconsistent layouts, scanned PDFs, multi-page tables, and non-standard formatting
- Why text extraction alone is not enough for downstream systems
- How structured outputs like JSON and CSV reduce post-processing work

Clearly explain how DocumentLens helps:
- Converts PDF table content into clean structured data
- Preserves row-column relationships and headers
- Handles scanned and image-based PDFs through visual document understanding
- Provides enterprise APIs for integration into existing pipelines
- Supports downstream workflows such as ERP ingestion, reconciliation, and analytics

Include practical examples such as invoices, bank statements, shipping docs, and compliance reports.
""",
        "keywords": [
            "PDF table extraction",
            "PDF table extraction Python",
            "PDF to JSON table extraction",
            "extract table from scanned PDF",
            "document parsing API",
            "Document AI API"
        ]
    },
    {
        "query": "Scanned PDF Data Extraction: Solving the Messiest Document Automation Problem",
        "instructions": """
Focus on scanned PDFs as a major pain point in real enterprise workflows.

Cover:
- Why scanned PDFs are harder than digital PDFs
- Issues such as skew, blur, low resolution, shadows, stamps, handwriting, and compression artifacts
- Why traditional OCR produces unreliable results from scanned documents
- Why businesses still need structured data from scanned contracts, invoices, receipts, forms, and government documents

Clearly explain how DocumentLens helps:
- Uses visual understanding to interpret scanned pages beyond raw OCR
- Applies layout-aware parsing to recover structure from image-based documents
- Extracts fields, tables, stamps, signatures, and handwritten notes where possible
- Grounds outputs to page locations for human verification
- Produces structured outputs ready for downstream systems

Position DocumentLens as suitable for high-volume scanned PDF workflows in finance, legal, insurance, healthcare, and government.
""",
        "keywords": [
            "scanned PDF data extraction",
            "extract data from scanned PDF",
            "scanned PDF OCR",
            "AI document extraction scanned PDFs",
            "PDF to structured data",
            "DocumentLens scanned documents"
        ]
    },
    {
        "query": "PDF to Structured Data: The Missing Link Between Documents and Automation",
        "instructions": """
Explain why converting PDFs into structured data is more valuable than simply extracting text.

Cover:
- Why PDFs are difficult for automation systems
- The difference between raw text, parsed content, and structured data
- Common structured output formats: JSON, CSV, XML, Markdown
- Why business systems need field names, tables, relationships, confidence scores, and source grounding

Clearly explain how DocumentLens helps:
- Converts PDFs into structured outputs based on user-defined schemas
- Preserves layout, tables, fields, and document hierarchy
- Supports both document extraction and full document parsing workflows
- Provides API-ready outputs for ERP, CRM, BI, IDP, and RPA systems
- Reduces manual cleanup and data mapping work

Position DocumentLens as the bridge from static PDF files to operational business data.
""",
        "keywords": [
            "PDF to structured data",
            "convert PDF to JSON",
            "PDF data extraction AI",
            "document parsing",
            "AI document extraction",
            "structured data extraction"
        ]
    },
    {
        "query": "Document Fraud Detection: Building Trust into Digital Workflows",
        "instructions": """
Focus on why document fraud detection is becoming important in document-heavy workflows.

Cover:
- Common fraud risks: altered images, edited invoices, forged IDs, modified receipts, tampered contracts
- Why visual manipulation is hard to detect manually at scale
- Why OCR alone cannot detect forgery because it reads content but does not assess image integrity
- High-risk industries: finance, insurance, legal, logistics, procurement, and government

Clearly explain how DocumentLens helps:
- Complements extraction with image forgery detection capabilities
- Flags suspicious visual inconsistencies for review
- Supports document verification workflows before data enters downstream systems
- Can work alongside extraction, parsing, and comparison services
- Helps teams reduce manual review burden while improving trust

Position DocumentLens as part of a broader document trust layer, not just an OCR tool.
""",
        "keywords": [
            "document fraud detection",
            "AI document fraud detection",
            "image forgery detection",
            "forged invoice detection",
            "document verification AI",
            "fraud detection document AI"
        ]
    },
    {
        "query": "Shipping Document AI: Automating the Paperwork Behind Global Logistics",
        "instructions": """
Focus on logistics and supply chain paperwork.

Cover:
- Common shipping documents: bills of lading, packing lists, commercial invoices, delivery orders, customs declarations
- Why logistics documents are hard: many formats, stamps, signatures, tables, multilingual fields, and poor scan quality
- How manual processing causes delays, reconciliation issues, and customs clearance bottlenecks

Clearly explain how DocumentLens helps:
- Extracts structured data from shipping documents
- Preserves table and line-item relationships
- Detects stamps, signatures, and official markings
- Supports multilingual Southeast Asian logistics documents
- Outputs structured data for TMS, ERP, customs, and supply chain systems

Position DocumentLens as a way to reduce shipping delays and improve document-driven logistics visibility.
""",
        "keywords": [
            "shipping document AI",
            "AI document processing supply chain",
            "logistics document automation",
            "shipping document automation",
            "Document AI logistics use cases",
            "supply chain document automation AI"
        ]
    },
    {
        "query": "Bill of Lading OCR: Extracting Reliable Data from Complex Shipping Documents",
        "instructions": """
Focus specifically on bills of lading.

Cover:
- What fields matter: shipper, consignee, vessel, voyage, container number, seal number, goods description, weight, ports, dates
- Why bills of lading are challenging: inconsistent layouts, stamps, signatures, multi-page attachments, scanned copies
- Why basic OCR struggles with tables, layout, and field relationships

Clearly explain how DocumentLens helps:
- Uses layout-aware parsing to detect key sections and table structures
- Extracts bill of lading fields into structured outputs
- Handles scanned and multilingual shipping documents
- Grounds extracted data to the original page for verification
- Integrates with logistics, customs, and ERP systems through APIs

Position DocumentLens as a stronger alternative to traditional bill of lading OCR.
""",
        "keywords": [
            "bill of lading OCR",
            "AI for bills of lading",
            "bill of lading data extraction",
            "shipping document AI",
            "automated shipping document extraction",
            "logistics document AI"
        ]
    },
    {
        "query": "Enterprise Document Processing at Scale: From Thousands to Millions of Pages",
        "instructions": """
Focus on enterprise-scale document operations.

Cover:
- Why small OCR workflows break at scale
- Challenges with throughput, accuracy consistency, monitoring, human review queues, and downstream integration
- Why enterprises need more than a one-off extraction tool
- Examples from banking, insurance, logistics, healthcare, government, and education

Clearly explain how DocumentLens helps:
- Supports structured extraction and parsing through API workflows
- Produces consistent outputs across large document volumes
- Uses traceability and confidence signals to reduce manual review
- Can support cloud and enterprise deployment patterns
- Handles diverse documents including scanned PDFs, tables, stamps, handwriting, and multilingual files

Position DocumentLens as an enterprise document intelligence layer built for operational scale.
""",
        "keywords": [
            "enterprise document processing at scale",
            "enterprise document AI",
            "high volume document processing",
            "Document AI API",
            "intelligent document processing enterprise",
            "document automation at scale"
        ]
    },
    {
        "query": "Intelligent Document Processing for Insurance: Claims, Policies, and Evidence",
        "instructions": """
Focus on insurance workflows.

Cover:
- Key document types: claims forms, policy schedules, incident reports, medical reports, receipts, identity documents, photos
- Why insurance document processing is difficult: inconsistent attachments, handwritten notes, stamps, scanned PDFs, multilingual forms
- Impact of manual processing: slow claims, errors, fraud risk, poor customer experience

Clearly explain how DocumentLens helps:
- Extracts structured claim and policy data
- Parses supporting documents and evidence
- Detects visual elements such as stamps, signatures, and figures
- Supports multilingual forms and regional formats
- Grounds every field to source locations for review
- Enables downstream automation for claim intake, routing, and verification

Position DocumentLens as a practical IDP solution for insurers that need speed and accuracy.
""",
        "keywords": [
            "intelligent document processing for insurance",
            "Document AI insurance claims",
            "insurance claims automation AI",
            "insurance document classification with AI",
            "claims intake automation using Document AI",
            "enterprise document processing for insurers"
        ]
    },
    {
        "query": "OCR vs Document AI: What Enterprises Need to Know Before Automating",
        "instructions": """
Write an educational comparison article.

Cover:
- What OCR does well: character recognition and basic digitization
- Where OCR fails: structure, context, tables, handwriting, stamps, multilingual documents, and downstream usability
- What Document AI adds: layout understanding, semantic extraction, structured output, source grounding, and API integration
- When OCR is enough vs when Document AI is required

Clearly explain how DocumentLens fits:
- Goes beyond OCR with layout analysis and VLM-powered understanding
- Extracts fields, tables, stamps, figures, and structured layouts
- Produces JSON/CSV/XML outputs for enterprise workflows
- Supports Southeast Asian languages and regional document formats

Position DocumentLens as the right step when businesses need usable data, not just text.
""",
        "keywords": [
            "OCR vs Document AI",
            "AI document processing vs traditional OCR",
            "intelligent document processing",
            "Document AI use cases",
            "OCR limitations",
            "DocumentLens"
        ]
    },
    {
        "query": "OCR Accuracy in the Real World: Character Accuracy Is Not Business Accuracy",
        "instructions": """
Focus on the misleading nature of OCR accuracy claims.

Cover:
- Difference between character-level accuracy and field-level accuracy
- Examples where text is recognized correctly but placed in the wrong field
- Why table structure, context, and source grounding matter
- Why businesses care about usable data, not OCR benchmark numbers

Clearly explain how DocumentLens helps:
- Focuses on structured field extraction and semantic correctness
- Preserves layout and document hierarchy
- Grounds values to their original locations for verification
- Uses confidence and schema-aligned outputs to improve downstream reliability

Position DocumentLens as a business-accuracy-focused document AI system.
""",
        "keywords": [
            "OCR accuracy",
            "OCR benchmark",
            "field extraction accuracy",
            "AI document extraction accuracy",
            "document extraction validation",
            "Document AI accuracy"
        ]
    },
    {
        "query": "Receipt OCR for Real Expense Workflows: Beyond Simple Text Capture",
        "instructions": """
Focus on receipt processing pain points.

Cover:
- Real receipts are messy: faded thermal paper, crumpled images, handwriting, multiple languages, missing fields
- Important fields: merchant, date, tax, currency, total, payment method, line items
- Why traditional OCR creates manual cleanup work

Clearly explain how DocumentLens helps:
- Extracts receipt data as structured fields
- Handles scanned or photographed receipts
- Supports multilingual and regional receipts, especially Southeast Asia
- Preserves source grounding for verification
- Outputs data ready for expense, accounting, and audit systems

Position DocumentLens as a practical solution for expense automation and receipt intelligence.
""",
        "keywords": [
            "receipt OCR",
            "receipt data extraction",
            "expense receipt automation",
            "AI receipt extraction",
            "multilingual receipt OCR",
            "Document AI receipts"
        ]
    },
    {
        "query": "Invoice OCR for Complex Regional Documents: Accuracy Without Template Maintenance",
        "instructions": """
Focus on invoice extraction in regional and multilingual contexts.

Cover:
- Invoice diversity: layouts, tax fields, supplier formats, item tables, currencies, stamps
- Why template-based OCR becomes expensive to maintain
- Challenges in Southeast Asian invoices: multilingual content, local formats, regional tax documents

Clearly explain how DocumentLens helps:
- Extracts supplier, buyer, invoice number, tax, totals, line items, and currency
- Uses layout-aware analysis instead of fixed templates
- Supports SEA-specific invoice formats and languages
- Outputs structured data for ERP/accounting systems
- Reduces manual review and template maintenance

Position DocumentLens as invoice intelligence, not just invoice OCR.
""",
        "keywords": [
            "invoice OCR",
            "automated invoice extraction",
            "AI invoice processing",
            "invoice data extraction",
            "multilingual invoice OCR",
            "Document AI invoice processing"
        ]
    },
    {
        "query": "Handwritten Text Recognition Challenges in Enterprise Documents",
        "instructions": """
Focus on handwritten text as a persistent document AI challenge.

Cover:
- Why handwriting is difficult: cursive, inconsistent spacing, mixed print-handwriting, poor scans, abbreviations
- Common enterprise cases: forms, receipts, medical notes, education assignments, government archives, claim annotations
- Why traditional OCR performs poorly on handwriting

Clearly explain how DocumentLens helps:
- Treats handwriting as part of the visual document, not noise
- Uses context-aware understanding to interpret handwritten fields
- Combines layout, surrounding text, and semantic reasoning
- Grounds uncertain outputs for review
- Supports workflows where handwriting is mixed with printed content

Position DocumentLens as suitable for challenging handwritten document extraction scenarios.
""",
        "keywords": [
            "challenges in handwritten text recognition",
            "handwriting OCR",
            "AI handwriting recognition",
            "handwritten document extraction",
            "scanned handwritten forms",
            "Document AI handwriting"
        ]
    },
    {
        "query": "Expense Receipt Recognition Accuracy: What Finance Teams Should Measure",
        "instructions": """
Focus on expense receipt recognition accuracy from a business perspective.

Cover:
- Why receipt OCR accuracy should be measured by field correctness, not just text recognition
- Key fields: amount, tax, merchant, date, currency, category, payment method
- Common failure cases: wrong total, local currency confusion, missing tax, multiple totals on one receipt
- How low accuracy creates manual rework and audit issues

Clearly explain how DocumentLens helps:
- Extracts receipt fields into structured outputs
- Uses semantic reasoning to identify the correct total and currency
- Supports multilingual and regional receipts
- Provides traceability for audit and correction
- Enables integration into expense and accounting workflows

Position DocumentLens as a higher-quality extraction layer for finance teams.
""",
        "keywords": [
            "expensepro receipt recognition accuracy",
            "receipt recognition accuracy",
            "expense receipt OCR",
            "AI expense automation",
            "receipt data extraction accuracy",
            "expense document AI"
        ]
    },
    {
        "query": "Reducing Shipping Delays with AI-Driven Document Processing",
        "instructions": """
Focus on how document issues delay logistics operations.

Cover:
- Shipping delays caused by missing or incorrect paperwork
- Common documents: bill of lading, customs declaration, packing list, commercial invoice, certificates
- Manual checking bottlenecks and risk of incorrect declarations
- Multilingual and cross-border document complexity

Clearly explain how DocumentLens helps:
- Extracts shipping data from diverse document formats
- Preserves tables and line items
- Detects stamps, signatures, and official fields
- Supports multilingual trade documentation
- Integrates with logistics and customs systems
- Enables earlier validation and faster clearance

Position DocumentLens as a document intelligence tool for reducing logistics bottlenecks.
""",
        "keywords": [
            "ai-driven document processing shipping delays reduction",
            "shipping document AI",
            "Document AI customs",
            "bill of lading OCR",
            "logistics document automation",
            "supply chain document automation AI"
        ]
    },
    {
        "query": "Manual vs Automated Contract Management: Risks Hidden in Document Review",
        "instructions": """
Focus on contract management risks from manual review.

Cover:
- Common manual contract management risks: missed renewal dates, overlooked obligations, inconsistent clause review, slow due diligence
- Why OCR alone cannot solve contract review because it loses clause hierarchy and context
- Challenges with scanned contracts, signatures, stamps, amendments, and multilingual contracts

Clearly explain how DocumentLens helps:
- Parses contract structure and clause hierarchy
- Extracts parties, dates, values, obligations, renewal terms, and signatures
- Grounds extracted clauses and fields to source locations
- Supports structured outputs for CLM, legal review, and compliance systems
- Enables faster due diligence and contract monitoring

Position DocumentLens as legal document intelligence rather than simple OCR.
""",
        "keywords": [
            "manual vs automated contract management risks",
            "AI contract review",
            "contract data extraction AI",
            "legal document automation",
            "Document AI legal contracts",
            "contract intelligence"
        ]
    },
    {
        "query": "Multilingual OCR for Southeast Asia: From Text Recognition to Context Understanding",
        "instructions": """
Focus on multilingual document automation in Southeast Asia.

Cover:
- Common language challenges: Vietnamese, Bahasa, Tagalog, Hindi, Thai, mixed English, and local terms
- Why multilingual OCR struggles with language switching, local formats, and names
- Why cultural context matters in business documents

Clearly explain how DocumentLens helps:
- Supports regional language document processing
- Understands local naming conventions and document hierarchies
- Handles mixed-language pages with layout and semantic understanding
- Extracts structured data from invoices, receipts, forms, financial documents, and legal documents

Position DocumentLens as purpose-built document intelligence for Southeast Asia.
""",
        "keywords": [
            "multilingual OCR",
            "multilingual document AI Southeast Asia",
            "Document AI ASEAN",
            "regional language document processing",
            "Vietnamese OCR",
            "Bahasa document AI",
            "Tagalog OCR"
        ]
    },
    {
        "query": "AI Document Extraction for Real Business Workflows: From Upload to API Output",
        "instructions": """
Focus on end-to-end AI document extraction.

Cover:
- Business need to extract key fields from documents and push them into systems
- Difference between basic OCR, key-value extraction, structured extraction, and full parsing
- Common blockers: layout complexity, low-quality scans, multilingual formats, and inconsistent schemas

Clearly explain how DocumentLens helps:
- Allows users to upload documents and define schemas
- Extracts context-aware structured data
- Supports key-value extraction, table extraction, layout extraction, and parsing
- Provides enterprise APIs for integration into workflows
- Supports Southeast Asian document types and languages

Position DocumentLens as an end-to-end document extraction platform, not a one-off OCR utility.
""",
        "keywords": [
            "AI document extraction",
            "document extraction API",
            "key value extraction",
            "structured data extraction",
            "AI document processing",
            "DocumentLens extraction"
        ]
    },
    {
        "query": "Document Parsing for AI Agents: Preparing PDFs for Reliable Reasoning",
        "instructions": """
Focus on document parsing as a foundation for AI agents and RAG workflows.

Cover:
- Why AI agents need structured document inputs
- Problems with raw OCR text: broken order, missing tables, lost figures, no hierarchy
- How poor parsing leads to hallucinations and weak retrieval

Clearly explain how DocumentLens helps:
- Parses documents into structured Markdown/JSON
- Preserves headings, sections, tables, and layout relationships
- Handles complex and multilingual PDFs
- Grounds parsed content to source pages
- Provides cleaner inputs for AI agents, search, and knowledge workflows

Position DocumentLens as infrastructure for document-aware AI systems.
""",
        "keywords": [
            "document parsing",
            "PDF parsing for LLM",
            "document parsing API",
            "PDF to Markdown",
            "RAG document parsing",
            "AI agent document processing"
        ]
    },
    {
        "query": "Watermark Cleanup for Document AI: Improving Extraction from Noisy PDFs",
        "instructions": """
Focus on how watermarks and background noise harm document extraction.

Cover:
- Common noise sources: watermarks, stamps, shadows, background patterns, low contrast, scan artifacts
- Why OCR accuracy drops when text overlaps with visual noise
- Business impact: missing fields, wrong values, manual rework

Clearly explain how DocumentLens helps:
- Includes watermark cleanup and noise reduction capabilities
- Improves readability before extraction
- Helps downstream layout analysis and VLM understanding
- Supports scanned PDFs, invoices, contracts, and forms

Position watermark cleanup as a practical preprocessing step inside a robust document intelligence workflow.
""",
        "keywords": [
            "watermark cleanup",
            "remove watermark from scanned PDF",
            "OCR noisy PDF",
            "scanned PDF data extraction",
            "document image enhancement",
            "AI document processing"
        ]
    },
    {
        "query": "Stamp Detection in Document AI: Capturing What OCR Ignores",
        "instructions": """
Focus on stamps and official markings.

Cover:
- Why stamps matter in legal, finance, logistics, government, and procurement documents
- Why traditional OCR ignores stamps or treats them as visual noise
- Consequences of losing stamp information: incomplete records, weak verification, audit gaps

Clearly explain how DocumentLens helps:
- Detects stamps, seals, and official markings
- Links stamps to relevant document fields and page regions
- Preserves them as part of structured outputs
- Supports workflows that need verification and audit traceability

Position DocumentLens as a multimodal document intelligence tool that captures text and visual evidence.
""",
        "keywords": [
            "stamp detection",
            "seal detection document AI",
            "document fraud detection",
            "official stamp OCR",
            "multimodal document AI",
            "DocumentLens stamp detection"
        ]
    },
    {
        "query": "Chart and Figure Analysis in Documents: Extracting Insights Beyond Text",
        "instructions": """
Focus on charts, figures, diagrams, and visual content inside documents.

Cover:
- Why OCR ignores charts and figures
- Why charts matter in reports, financial documents, medical records, education worksheets, and research documents
- Challenges: legends, axes, labels, embedded images, screenshots, and mixed text-image content

Clearly explain how DocumentLens helps:
- Analyzes charts, graphs, and figures as visual information
- Converts visual elements into structured insights or natural language descriptions
- Preserves position and context in the document
- Supports downstream analytics, review, and automation

Position DocumentLens as capable of understanding documents as visual + textual artifacts.
""",
        "keywords": [
            "chart and figure analysis",
            "AI figure extraction",
            "extract charts from PDF",
            "document image understanding",
            "multimodal document processing",
            "Document AI figures"
        ]
    },
    {
        "query": "Key-Value Extraction from Complex Forms Without Fixed Templates",
        "instructions": """
Focus on key-value extraction from forms and semi-structured documents.

Cover:
- What key-value extraction means in business workflows
- Why fixed templates fail when field positions vary
- Challenges with multi-page forms, checkboxes, handwritten values, and mixed-language labels

Clearly explain how DocumentLens helps:
- Detects labels and values using layout and semantic relationships
- Handles varied form layouts without fixed coordinate templates
- Grounds each key-value pair to the source page
- Outputs schema-aligned structured data
- Supports enterprise workflows like KYC, insurance claims, HR forms, healthcare intake, and government forms

Position DocumentLens as flexible key-value extraction for real-world forms.
""",
        "keywords": [
            "key value extraction",
            "form data extraction",
            "AI form extraction",
            "document extraction API",
            "structured data extraction",
            "DocumentLens key value extraction"
        ]
    },
    {
        "query": "Structured Data Extraction from Invoices, Forms, and Tables",
        "instructions": """
Focus on structured data extraction as a step beyond OCR.

Cover:
- Difference between extracting text and extracting structured datasets
- Examples: invoice line items, insurance claims, supplier records, bank statements, customs forms
- Why downstream systems need consistent fields, types, and relationships

Clearly explain how DocumentLens helps:
- Extracts related fields as structured records
- Preserves tables, line items, and nested fields
- Supports schema-based extraction
- Outputs JSON/CSV/XML ready for ERP, CRM, BI, or data warehouse ingestion
- Provides traceability and confidence for review

Position DocumentLens as a structured extraction engine for operational data pipelines.
""",
        "keywords": [
            "structured data extraction",
            "PDF to structured data",
            "AI document extraction",
            "table extraction from PDF",
            "invoice data extraction",
            "enterprise document processing"
        ]
    },
    {
        "query": "Layout Extraction for Complex PDFs: Preserving the Structure OCR Loses",
        "instructions": """
Focus on layout extraction as a critical document intelligence capability.

Cover:
- What layout extraction means: sections, headings, columns, tables, checkboxes, figures, forms
- Why traditional OCR loses layout and reading order
- How layout loss breaks summaries, field extraction, and automation

Clearly explain how DocumentLens helps:
- Captures complex layouts including figures, forms, checkboxes, and multi-column structures
- Preserves reading order and document hierarchy
- Helps extraction, parsing, comparison, and downstream AI reasoning
- Supports scanned and digital documents
- Provides structured outputs with page-level context

Position layout extraction as one of DocumentLens' core enterprise API capabilities.
""",
        "keywords": [
            "layout extraction",
            "PDF layout analysis",
            "document layout analysis",
            "multi-column PDF extraction",
            "document parsing",
            "Document AI layout"
        ]
    },
    {
        "query": "Document Comparison for Contracts, Policies, and Compliance Reviews",
        "instructions": """
Focus on document comparison as a practical enterprise workflow.

Cover:
- Common comparison needs: contract versions, policy changes, compliance documents, legal updates, supplier terms
- Why text diff tools fail when documents are scanned, reformatted, or structurally changed
- Risks of missing semantic changes in regulated workflows

Clearly explain how DocumentLens helps:
- Compares documents at structural and semantic levels
- Highlights insertions, deletions, and modifications
- Supports scanned vs digital comparisons where possible
- Grounds changes to page locations
- Helps legal, compliance, procurement, and audit teams review faster

Position DocumentLens document comparison as a higher-confidence alternative to manual review and basic text diff.
""",
        "keywords": [
            "document comparison AI",
            "contract comparison AI",
            "compare scanned documents",
            "legal document comparison",
            "policy document comparison",
            "document intelligence"
        ]
    },
    {
        "query": "Image Forgery Detection for Receipts, Invoices, and Claims Documents",
        "instructions": """
Focus on image forgery detection in business document workflows.

Cover:
- Common manipulated documents: receipts, invoices, claim photos, ID scans, contracts, delivery proofs
- Why manual review is unreliable at scale
- Why OCR cannot detect manipulation because it only reads text
- High-risk workflows: insurance claims, procurement, reimbursement, compliance, KYC

Clearly explain how DocumentLens helps:
- Offers image forgery detection as part of the broader document suite
- Flags suspicious visual inconsistencies and possible tampering
- Complements extraction, parsing, and comparison workflows
- Helps teams review questionable documents before data enters downstream systems

Position DocumentLens as a trust and verification layer for document-heavy operations.
""",
        "keywords": [
            "image forgery detection",
            "document fraud detection",
            "receipt fraud detection",
            "invoice fraud detection AI",
            "insurance document fraud detection AI",
            "document verification AI"
        ]
    },
    {
        "query": "Southeast Asian Invoice Processing: Handling Local Formats, Languages, and Tax Fields",
        "instructions": """
Focus on invoices across Southeast Asia.

Cover:
- Regional diversity in invoice formats, tax identifiers, currencies, languages, and supplier layouts
- Why generic invoice OCR underperforms on local documents
- Issues with mixed English and local languages, stamps, and scanned invoices

Clearly explain how DocumentLens helps:
- Is tailored for Southeast Asian document formats
- Supports regional language processing and cultural context
- Extracts invoice numbers, supplier names, tax IDs, totals, line items, and currency
- Preserves layout and table structures
- Outputs ERP-ready structured data

Position DocumentLens as purpose-built for SEA invoice intelligence.
""",
        "keywords": [
            "invoice OCR Southeast Asia",
            "multilingual invoice OCR",
            "regional invoice document AI",
            "Document AI ASEAN invoices",
            "AI invoice processing",
            "DocumentLens Southeast Asia"
        ]
    },
    {
        "query": "Vietnamese Document OCR: From Characters to Context-Aware Extraction",
        "instructions": """
Focus specifically on Vietnamese documents.

Cover:
- Vietnamese diacritics, local invoice formats, receipts, forms, and business documents
- Why generic OCR may misread accented text or lose field context
- Mixed-language documents with Vietnamese and English

Clearly explain how DocumentLens helps:
- Supports Vietnamese document processing with context awareness
- Preserves layout and field relationships
- Extracts structured data from invoices, forms, receipts, and reports
- Handles regional business conventions
- Provides API-ready outputs for enterprise workflows

Position DocumentLens as Vietnamese document intelligence, not just Vietnamese OCR.
""",
        "keywords": [
            "Vietnamese OCR",
            "Vietnamese document AI",
            "Vietnamese invoice OCR",
            "multilingual OCR",
            "Southeast Asia document intelligence",
            "DocumentLens Vietnamese"
        ]
    },
    {
        "query": "Bahasa Document AI for Invoices, Forms, and Regional Business Workflows",
        "instructions": """
Focus on Bahasa Indonesia/Malay document processing.

Cover:
- Local business documents, receipts, invoices, tax forms, and procurement files
- Language and formatting challenges
- Mixed-language fields and local naming conventions

Clearly explain how DocumentLens helps:
- Supports Bahasa documents with layout and semantic understanding
- Extracts structured key fields from local business documents
- Handles table-heavy invoices and forms
- Preserves source grounding for review
- Integrates with finance and operations workflows

Position DocumentLens as a regional document AI system for Bahasa-speaking markets.
""",
        "keywords": [
            "Bahasa OCR",
            "Bahasa document AI",
            "Malay OCR",
            "Indonesian invoice OCR",
            "multilingual document AI Southeast Asia",
            "regional language document processing"
        ]
    },
    {
        "query": "Tagalog Document Processing for Local Forms, Receipts, and Business Documents",
        "instructions": """
Focus on Tagalog and Philippine business document processing.

Cover:
- Philippine forms, receipts, invoices, BIR-related documents, and mixed English/Tagalog content
- Challenges with local document types and regional wording
- Why generic OCR lacks contextual awareness

Clearly explain how DocumentLens helps:
- Is pre-trained on regional document types including Philippine forms
- Supports Tagalog and English mixed documents
- Extracts structured fields from forms, receipts, invoices, and government-related documents
- Preserves layout and source traceability

Position DocumentLens as a strong fit for Philippine document workflows.
""",
        "keywords": [
            "Tagalog OCR",
            "Philippine document AI",
            "BIR forms OCR",
            "multilingual OCR Philippines",
            "Southeast Asia document AI",
            "DocumentLens Tagalog"
        ]
    },
    {
        "query": "Hindi Document OCR for Forms, Receipts, and Business Records",
        "instructions": """
Focus on Hindi document processing.

Cover:
- Challenges with Hindi script recognition, scanned forms, handwritten notes, and mixed English/Hindi documents
- Why global OCR tools often underperform on regional scripts and layouts
- Use cases in banking, insurance, healthcare, government, and retail

Clearly explain how DocumentLens helps:
- Supports Hindi document extraction with layout-aware analysis
- Extracts structured fields from forms, receipts, invoices, and reports
- Handles mixed English and Hindi content
- Grounds results to source page positions
- Outputs structured data for enterprise systems

Position DocumentLens as a regional document intelligence system for Hindi content.
""",
        "keywords": [
            "Hindi OCR",
            "Hindi document AI",
            "multilingual OCR",
            "Hindi form data extraction",
            "regional language OCR",
            "DocumentLens Hindi"
        ]
    },
    {
        "query": "AI Document Processing for KYC: Extracting Trustworthy Data from Regional Documents",
        "instructions": """
Focus on KYC document processing.

Cover:
- KYC document types: IDs, proof of address, bank statements, tax forms, corporate registration documents
- Challenges: multilingual documents, stamps, signatures, scans, inconsistent formats
- Why OCR errors cause compliance risk and onboarding delays

Clearly explain how DocumentLens helps:
- Extracts structured KYC data from mixed document types
- Handles regional language and document formats
- Grounds extracted fields for audit and compliance review
- Supports API integration into onboarding workflows
- Can complement fraud detection and image verification workflows

Position DocumentLens as an enterprise document AI layer for KYC automation.
""",
        "keywords": [
            "Document AI KYC automation",
            "KYC document extraction",
            "AML document automation",
            "bank compliance document automation",
            "multilingual document AI",
            "secure document AI API"
        ]
    },
    {
        "query": "Medical Claims Document AI: Extracting Data from Forms, Reports, and Evidence",
        "instructions": """
Focus on medical claims and healthcare-insurance workflows.

Cover:
- Document types: claim forms, medical reports, invoices, lab results, discharge summaries, receipts
- Challenges: handwriting, medical abbreviations, scanned documents, multilingual forms, attachments
- Why manual review slows claims processing

Clearly explain how DocumentLens helps:
- Extracts structured claim and medical data
- Parses tables, forms, and supporting evidence
- Handles handwritten and printed content where possible
- Grounds results for verification
- Integrates with insurance claims and healthcare admin systems

Position DocumentLens as a tool for faster and more reliable medical claims processing.
""",
        "keywords": [
            "Document AI medical claims",
            "AI for patient document extraction",
            "automated medical report extraction",
            "healthcare documentation AI",
            "insurance claims automation AI",
            "medical document extraction"
        ]
    },
    {
        "query": "Clinical Research Document Extraction: Structuring Data from Trial Forms and Reports",
        "instructions": """
Focus on clinical research documents.

Cover:
- Document types: consent forms, case report forms, lab reports, medical histories, study records
- Challenges: handwritten entries, tables, multi-page forms, scanned archives, compliance requirements
- Why structured extraction matters for trial operations and research workflows

Clearly explain how DocumentLens helps:
- Extracts structured data from complex research documents
- Preserves tables, signatures, and field relationships
- Grounds extracted values to source pages
- Supports secure processing and downstream analytics
- Reduces manual data entry burden

Position DocumentLens as suitable for healthcare and clinical research document intelligence.
""",
        "keywords": [
            "AI document extraction clinical research",
            "clinical document extraction",
            "healthcare document compliance AI",
            "automated medical report extraction",
            "document AI healthcare workflow",
            "structured clinical data extraction"
        ]
    },
    {
        "query": "Purchase Order Automation: Extracting Line Items, Suppliers, and Delivery Details",
        "instructions": """
Focus on purchase order document automation.

Cover:
- Key PO fields: supplier, buyer, PO number, delivery date, line items, quantities, unit prices, payment terms
- Challenges: supplier-specific formats, scanned POs, tables, multilingual documents, missing references
- Why PO automation is difficult with OCR alone

Clearly explain how DocumentLens helps:
- Extracts structured PO data and line-item tables
- Preserves row-column relationships
- Links related fields for downstream procurement workflows
- Handles supplier-specific and regional formats
- Outputs ERP-ready data for procurement and finance systems

Position DocumentLens as a practical purchase order extraction system for procurement automation.
""",
        "keywords": [
            "Document AI purchase orders",
            "purchase order automation",
            "PO data extraction",
            "procurement document automation",
            "supplier document AI",
            "Document AI ERP integration"
        ]
    },
    {
        "query": "Three-Way Matching Automation with Document AI: PO, Invoice, and Receipt",
        "instructions": """
Focus on three-way matching in finance and procurement.

Cover:
- What three-way matching requires: PO, invoice, goods receipt/delivery note
- Common problems: inconsistent supplier documents, missing PO numbers, table mismatches, currency and tax differences
- Why manual matching is slow and error-prone

Clearly explain how DocumentLens helps:
- Extracts structured data from POs, invoices, and delivery notes
- Preserves line items, quantities, prices, tax, and totals
- Enables matching logic in downstream ERP or procurement systems
- Handles multilingual and regional supplier documents
- Grounds fields for exception review

Position DocumentLens as the document extraction layer that makes three-way matching automation reliable.
""",
        "keywords": [
            "three-way matching automation",
            "PO invoice matching AI",
            "supplier invoice automation",
            "Document AI purchase orders",
            "automated invoice extraction",
            "procurement document automation"
        ]
    },
    {
        "query": "Customs Clearance Automation with Document AI for Cross-Border Trade",
        "instructions": """
Focus on customs and cross-border trade.

Cover:
- Customs document types: declarations, invoices, certificates of origin, packing lists, bills of lading
- Challenges: multilingual forms, stamps, official signatures, inconsistent templates, high compliance requirements
- How document errors delay clearance and increase cost

Clearly explain how DocumentLens helps:
- Extracts structured customs data
- Detects stamps and official markings
- Preserves tables and shipment details
- Supports multilingual regional documents
- Integrates with customs, logistics, and compliance workflows

Position DocumentLens as a tool to reduce clearance delays and improve trade document accuracy.
""",
        "keywords": [
            "Document AI customs",
            "automated customs documentation",
            "shipping document AI",
            "bill of lading OCR",
            "cross-border trade document automation",
            "logistics document AI"
        ]
    },
    {
        "query": "Policy Document Analysis for Insurance Operations",
        "instructions": """
Focus on insurance policy documents.

Cover:
- Policy schedules, endorsements, exclusions, riders, renewal notices, terms and conditions
- Challenges: long documents, dense tables, legal language, scanned copies, version updates
- Why manual policy review is slow and risky

Clearly explain how DocumentLens helps:
- Parses policy document structure
- Extracts key terms, coverage, exclusions, dates, premiums, and insured parties
- Supports document comparison for updated policy versions
- Grounds extracted fields to original pages
- Outputs structured data for policy servicing and compliance monitoring

Position DocumentLens as an insurance document intelligence tool.
""",
        "keywords": [
            "Document AI policy document analysis",
            "insurance document classification with AI",
            "enterprise document processing for insurers",
            "AI in insurance document processing",
            "policy document automation",
            "insurance document AI"
        ]
    },
    {
        "query": "Claims Intake Automation Using Document AI",
        "instructions": """
Focus on first-mile insurance claims intake.

Cover:
- Claim packets contain forms, photos, receipts, incident reports, medical records, and policy details
- Problems with manual intake: slow triage, missing documents, inconsistent classification, data entry errors
- Why OCR alone cannot classify and extract from mixed evidence

Clearly explain how DocumentLens helps:
- Classifies and parses multiple document types
- Extracts structured claim fields
- Handles receipts, reports, forms, and visual evidence
- Supports fraud/anomaly review through document verification capabilities
- Sends structured data downstream for routing and decisioning

Position DocumentLens as an intake automation layer for insurers.
""",
        "keywords": [
            "claims intake automation using Document AI",
            "Document AI insurance claims",
            "insurance claims automation AI",
            "insurance document fraud detection AI",
            "AI in insurance document processing",
            "enterprise document processing for insurers"
        ]
    },
    {
        "query": "Enterprise Document API Best Practices for Reliable Automation",
        "instructions": """
Focus on API integration best practices.

Cover:
- What enterprises need from a document processing API: uptime, structured outputs, error handling, confidence, traceability, security
- Why OCR APIs often return text that still requires heavy cleanup
- How to design workflows around ingestion, extraction, validation, and downstream delivery

Clearly explain how DocumentLens helps:
- Provides enterprise APIs for extraction and parsing
- Outputs structured JSON/CSV/XML/Markdown depending on use case
- Supports source grounding and confidence-aware review
- Handles complex layouts, tables, multilingual documents, and visual elements
- Fits into ERP, CRM, BI, IDP, and RPA workflows

Position DocumentLens as an API-first document intelligence platform.
""",
        "keywords": [
            "enterprise document API best practices",
            "Document AI API integration",
            "secure document AI API",
            "document processing API comparison",
            "enterprise document AI API",
            "Document AI CRM integration"
        ]
    },
    {
        "query": "Comparing Document Processing APIs: What Matters Beyond OCR",
        "instructions": """
Focus on evaluating document processing APIs.

Cover:
- Comparison criteria: layout preservation, table extraction, schema support, multilingual capability, traceability, security, scalability
- Why simple OCR APIs are insufficient for enterprise workflows
- What buyers should test using real documents

Clearly explain how DocumentLens differentiates:
- Offers extraction, parsing, comparison, and forgery detection services
- Supports advanced layout analysis and VLM-based understanding
- Handles Southeast Asian languages and regional documents
- Provides structured outputs and enterprise integration capability
- Grounds results to original pages for verification

Position DocumentLens as a strong choice for teams evaluating document AI APIs.
""",
        "keywords": [
            "best document AI APIs",
            "document processing API comparison",
            "Document AI API integration",
            "AI document extraction API",
            "enterprise document AI API",
            "secure document AI API"
        ]
    },
    {
        "query": "ERP Workflow Automation with Document AI: From Invoices to Approved Entries",
        "instructions": """
Focus on ERP automation.

Cover:
- Common ERP document inputs: invoices, purchase orders, receipts, delivery notes, vendor forms
- Manual bottlenecks before ERP entry
- Why poor extraction causes payment delays and reconciliation problems

Clearly explain how DocumentLens helps:
- Extracts ERP-ready structured data
- Preserves line items, totals, supplier details, and tax fields
- Validates and grounds extracted data before system entry
- Integrates through APIs into ERP workflows
- Supports multilingual supplier documents and regional invoices

Position DocumentLens as the document intelligence bridge into ERP systems.
""",
        "keywords": [
            "Document AI in ERP",
            "ERP document automation",
            "automated invoice extraction",
            "Document AI API integration",
            "supplier invoice automation",
            "PDF to structured data"
        ]
    },
    {
        "query": "CRM Document Automation: Turning Customer Files into Structured Intelligence",
        "instructions": """
Focus on CRM-connected document workflows.

Cover:
- Customer documents: application forms, onboarding files, contracts, IDs, support attachments, claims
- Why CRM teams struggle with unstructured document uploads
- Impact on onboarding, sales operations, customer support, and compliance

Clearly explain how DocumentLens helps:
- Extracts customer-related fields from diverse documents
- Parses forms, contracts, and attachments
- Provides structured outputs for CRM records
- Supports multilingual and regional documents
- Grounds extracted data for verification before updating records

Position DocumentLens as a document intelligence API for customer data automation.
""",
        "keywords": [
            "Document AI CRM integration",
            "Document AI API integration",
            "customer document automation",
            "AI document extraction",
            "enterprise document intelligence",
            "CRM workflow automation"
        ]
    },
    {
        "query": "Government Document Digitization: From Scanned Archives to Searchable Data",
        "instructions": """
Focus on public sector archives and government documents.

Cover:
- Challenges with old government archives: scans, handwriting, missing pages, multilingual records, stamps, seals
- Why digitization is not enough if documents remain unstructured images
- Need for searchable, structured, and traceable data

Clearly explain how DocumentLens helps:
- Extracts structured fields from scanned archive documents
- Handles handwriting and low-quality scans where possible
- Preserves source grounding for audit and review
- Supports multilingual documents and regional formats
- Converts static archives into searchable data assets

Position DocumentLens as useful for government digitization and historical archive modernization.
""",
        "keywords": [
            "government document digitization",
            "scanned PDF data extraction",
            "handwritten document extraction",
            "multilingual OCR",
            "document AI government",
            "PDF to structured data"
        ]
    },
    {
        "query": "Education Document AI: Parsing Student Assignments, Forms, and Records",
        "instructions": """
Focus on education use cases.

Cover:
- Documents: student assignments, exam sheets, enrollment forms, transcripts, handwritten answers, diagrams, tables
- Challenges: handwriting, printed text mixed with answers, no fixed template, images and charts needing descriptions
- Why OCR alone cannot separate questions, answers, and visual elements

Clearly explain how DocumentLens helps:
- Splits pages into structured modules
- Separates questions, answers, tables, images, and charts
- Converts visual student work into natural language descriptions where appropriate
- Returns element coordinates for review
- Supports cloud deployment and high-volume processing

Position DocumentLens as document intelligence for education technology and learning workflows.
""",
        "keywords": [
            "education document AI",
            "student assignment parsing",
            "handwritten text recognition challenges",
            "document parsing",
            "multimodal document AI",
            "AI document extraction education"
        ]
    },
    {
        "query": "Financial Statement Extraction: Turning Reports into Structured Analytics Data",
        "instructions": """
Focus on financial reports and statements.

Cover:
- Document types: balance sheets, income statements, cash flow reports, audit reports, annual reports
- Challenges: tables, footnotes, charts, multi-page layouts, scanned PDFs
- Why OCR alone fails to preserve numeric context and table relationships

Clearly explain how DocumentLens helps:
- Extracts financial tables and key figures
- Preserves table structure, headings, and notes
- Converts reports into structured data for analytics and BI
- Grounds extracted values to source pages
- Supports downstream financial analysis and audit workflows

Position DocumentLens as a document intelligence layer for financial analytics.
""",
        "keywords": [
            "financial statement extraction",
            "table extraction from PDF",
            "PDF table extraction",
            "AI for financial document analysis",
            "PDF to structured data",
            "enterprise document AI finance"
        ]
    },
    {
        "query": "Bank Statement Extraction for Reconciliation and Fraud Review",
        "instructions": """
Focus on bank statement processing.

Cover:
- Key fields: account holder, account number, transaction date, description, debit, credit, balance
- Challenges: multi-page tables, different bank formats, scanned statements, multilingual descriptions
- Why wrong table extraction breaks reconciliation

Clearly explain how DocumentLens helps:
- Preserves transaction tables and row relationships
- Extracts structured transaction data
- Supports multi-format and multilingual statements
- Enables downstream reconciliation, analytics, and fraud review
- Grounds transactions to original pages for verification

Position DocumentLens as a robust bank statement extraction system for finance teams.
""",
        "keywords": [
            "AI for bank statement analysis",
            "bank statement data extraction",
            "PDF table extraction",
            "financial document AI",
            "fraud detection document AI",
            "bank compliance document automation"
        ]
    },
    {
        "query": "Document Intelligence for Retail and E-Commerce Operations",
        "instructions": """
Focus on retail and e-commerce workflows.

Cover:
- Documents: supplier invoices, receipts, packing slips, return forms, customer claims, onboarding documents
- Challenges: high volume, inconsistent supplier formats, multilingual receipts, delivery notes, line-item tables
- Why manual processing slows operations

Clearly explain how DocumentLens helps:
- Extracts line items, PO numbers, supplier names, totals, and dates
- Processes receipts and packing slips
- Supports supplier onboarding documents and compliance records
- Integrates with ERP, inventory, and accounting systems
- Handles regional documents across Southeast Asia

Position DocumentLens as automation infrastructure for retail back-office workflows.
""",
        "keywords": [
            "retail document AI",
            "receipt OCR",
            "supplier invoice automation",
            "packing slip data extraction",
            "Document AI purchase orders",
            "supply chain document automation AI"
        ]
    },
    {
        "query": "Automated Packing Slip Extraction for Inventory and Delivery Verification",
        "instructions": """
Focus on packing slips.

Cover:
- Important fields: supplier, shipment ID, SKU, item description, quantity, delivery date, warehouse reference
- Challenges: line-item tables, scanned pages, inconsistent supplier templates, stamps, signatures
- Why packing slip extraction matters for inventory and fulfillment accuracy

Clearly explain how DocumentLens helps:
- Extracts structured item-level data from packing slips
- Preserves line-item tables
- Links packing slips to POs and invoices through shared fields
- Supports delivery verification and inventory updates
- Outputs data for ERP/WMS systems

Position DocumentLens as a practical tool for supply chain document automation.
""",
        "keywords": [
            "packing slip data extraction",
            "supply chain document automation AI",
            "Document AI logistics use cases",
            "supplier document automation",
            "ERP document automation",
            "logistics document AI"
        ]
    },
    {
        "query": "Data Extraction from Low-Quality Historical Documents",
        "instructions": """
Focus on old, degraded, or historical records.

Cover:
- Common issues: faded ink, torn pages, stains, handwritten text, old fonts, mixed languages, missing sections
- Why historical documents are hard for OCR
- Business and public-sector value of making archives searchable and structured

Clearly explain how DocumentLens helps:
- Applies visual and layout understanding to degraded documents
- Handles mixed printed and handwritten content where possible
- Extracts target fields with source traceability
- Supports human review where uncertainty remains
- Converts archives into searchable structured data

Position DocumentLens as suitable for archive digitization and challenging document intelligence tasks.
""",
        "keywords": [
            "historical document OCR",
            "scanned PDF data extraction",
            "handwritten document extraction",
            "low quality scan OCR",
            "government document digitization",
            "AI document extraction"
        ]
    },
    {
        "query": "Complex Document Layouts: Multi-Column PDFs, Footnotes, and Nested Tables",
        "instructions": """
Focus on complex layout problems.

Cover:
- Multi-column reading order problems
- Footnotes, sidebars, captions, headers, and nested tables
- Why OCR reads across columns incorrectly and produces broken outputs
- Impact on extraction, summarization, and downstream automation

Clearly explain how DocumentLens helps:
- Detects layout regions and preserves reading order
- Separates sections, tables, footnotes, captions, and sidebars
- Extracts structured content without mixing unrelated blocks
- Supports PDFs, scanned documents, reports, and legal/financial documents

Position DocumentLens as strong at complex layout understanding.
""",
        "keywords": [
            "complex document layout analysis",
            "multi-column PDF extraction",
            "PDF table extraction",
            "document parsing",
            "layout extraction",
            "OCR vs Document AI"
        ]
    },
    {
        "query": "Multi-Page Table Extraction from PDFs Without Losing Context",
        "instructions": """
Focus on tables that continue across pages.

Cover:
- Problems with multi-page tables: repeated headers, split rows, footnotes, totals on final page
- Why OCR and simple PDF extraction tools often break table continuity
- Business examples: bank statements, inventory logs, insurance schedules, financial reports

Clearly explain how DocumentLens helps:
- Detects table continuation across pages where applicable
- Preserves row-column relationships and header context
- Outputs clean structured data
- Grounds each row or cell to the original source page
- Supports downstream analytics and reconciliation

Position DocumentLens as useful for enterprise table-heavy document workflows.
""",
        "keywords": [
            "table extraction from PDF",
            "multi-page table extraction",
            "PDF table extraction",
            "extract tables from PDF",
            "financial table extraction",
            "PDF to structured data"
        ]
    },
    {
        "query": "Confidence Scores and Human Review Queues in Document AI",
        "instructions": """
Focus on operationalizing document AI.

Cover:
- Why automated extraction still needs quality controls
- How confidence scores help route uncertain fields to human review
- Problems with silent OCR errors
- How confidence-based workflows improve trust at scale

Clearly explain how DocumentLens helps:
- Provides confidence-aware extraction where applicable
- Grounds fields to source locations for fast review
- Supports quality gates before downstream system updates
- Helps reduce full-document review by focusing humans only on uncertain fields

Position DocumentLens as enabling practical enterprise automation with review controls.
""",
        "keywords": [
            "confidence score document AI",
            "OCR accuracy",
            "document extraction validation",
            "human in the loop document AI",
            "enterprise document processing at scale",
            "AI document extraction accuracy"
        ]
    },
    {
        "query": "Schema-Based Document Extraction: Getting the Fields Your Business Actually Needs",
        "instructions": """
Focus on schema-driven extraction.

Cover:
- Why different teams need different outputs from the same document
- Examples: finance needs totals and tax, legal needs dates and clauses, logistics needs shipment IDs
- Why generic OCR outputs too much irrelevant text

Clearly explain how DocumentLens helps:
- Allows users to define extraction requirements or schemas
- Extracts context-aware fields aligned to business needs
- Produces structured JSON/CSV/XML outputs
- Supports industry-specific workflows across banking, insurance, legal, healthcare, logistics, and retail
- Reduces downstream filtering and manual cleanup

Position DocumentLens as business-oriented document extraction, not generic text extraction.
""",
        "keywords": [
            "schema based document extraction",
            "AI document extraction",
            "key value extraction",
            "structured data extraction",
            "PDF to structured data",
            "enterprise document AI"
        ]
    }
]


In [4]:
output_folder = "content"
os.makedirs(output_folder, exist_ok=True)

print(f"Starting research on {len(research_tasks)} topics...")

# 4. Loop through the tasks
for i, task in enumerate(research_tasks, 1):
	query = task["query"]
	instructions = task["instructions"]
	keywords = task["keywords"] 

	full_instructions = f"{instructions}\n\n Use SEO keywords in your blog: {keywords}"
	
	print(f"[{i}/{len(research_tasks)}] Researching: {query}...")

	try:
		# Run the actual research
		# Note: We await here so it finishes one before starting the next
		report, sources = await basic_research(query, instructions=full_instructions)
		
		# Create a safe filename based on the query
		safe_name = clean_filename(query)
		file_path = os.path.join(output_folder, f"{safe_name}.md")

		# Save the report
		with open(file_path, "w", encoding="utf-8") as f:
			f.write(report)
		
		print(f"   Saved to: {file_path}")

	except Exception as e:
		print(f"   ERROR processing '{query}': {e}")

print("\nAll tasks completed.")

Starting research on 56 topics...
[1/56] Researching: Table Extraction from PDFs: Turning Broken Tables into Reliable Structured Data...
🔍 Starting research...

📚 Gathering information...

🔍 DEEP RESEARCH: Starting with breadth=2, depth=1, concurrency=4
Searching with Gemini Grounding: Table Extraction from PDFs: Turning Broken Tables into Reliable Structured Data
Resolving 10 Vertex AI redirect URLs to original sources...
Found 10 grounded results from Gemini.

📊 DEEP RESEARCH: depth=1, breadth=2, query=
        Initial Query: Table Extraction from PDFs: Turning Broken Tables into Reliable Structured D...
🔎 Generating 2 search queries...
✅ Generated 2 queries: ['"benchmark comparison 2026 multimodal LLMs vs specialized OCR tools for complex table extraction accuracy and cost"', '"state-of-the-art table structure recognition challenges 2026 evaluation metrics for multi-page nested tables"']


INFO:     [10:01:54] 🔍 Starting the research task for '"benchmark comparison 2026 multimodal LLMs vs specialized OCR tools for complex table extraction accuracy and cost"'...
INFO:     [10:01:54] 💻 Technology Analyst Agent
INFO:     [10:01:54] 🌐 Browsing the web to learn more about the task: "benchmark comparison 2026 multimodal LLMs vs specialized OCR tools for complex table extraction accuracy and cost"...


Searching with Gemini Grounding: "benchmark comparison 2026 multimodal LLMs vs specialized OCR tools for complex table extraction accuracy and cost"
Resolving 10 Vertex AI redirect URLs to original sources...


INFO:     [10:02:07] 🤔 Planning the research strategy and subtasks...
INFO:     [10:02:07] 🔍 Starting the research task for '"state-of-the-art table structure recognition challenges 2026 evaluation metrics for multi-page nested tables"'...
INFO:     [10:02:07] 🤖 AI Research Agent
INFO:     [10:02:07] 🌐 Browsing the web to learn more about the task: "state-of-the-art table structure recognition challenges 2026 evaluation metrics for multi-page nested tables"...


Found 10 grounded results from Gemini.
Searching with Gemini Grounding: "state-of-the-art table structure recognition challenges 2026 evaluation metrics for multi-page nested tables"
Resolving 10 Vertex AI redirect URLs to original sources...


...

Found 5 grounded results from Gemini.


I0000 00:00:1778667438.237426 212491168 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1778667438.363645 212491168 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1778667446.236813 212493682 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1778667446.407384 212493682 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
INFO:     [18:17:38] 📄 Scraped 3 pages of content
INFO:     [18:17:38] 🖼️ Selected 4 new images from 13 total images
INFO:     [18:17:38] 🌐 Scraping complete
INFO:     [18:17:38] 📚 Getting relevant content based on query: `LLM document extraction schema design trade-offs "prompt-based" vs "structured JSON" accuracy hallucination detection`...
INFO:     [18:17:39] 📚 Combined research context: 0 MCP sources, web content
INFO:     [18:17:39] Finalized research step.
💸 Total Rese

# Schema-Based Document Extraction: Getting the Fields Your Business Actually Needs

In today's data
-driven landscape, businesses are drowning in a sea of unstructured documents – from invoices and contracts to patient records and legal filings. The ability to efficiently extract precise, actionable data from these documents is no longer a luxury; it's an absolute necessity. Yet, many organizations still grapple with generic text extraction methods that yield an "Ambiguity Tax" of irrelevant information, high error rates, and endless manual cleanup. This is where **schema-based document extraction** emerges as a game-changer, allowing businesses to define exactly what data they need, ensuring outputs are not just valid but also perfectly aligned with their specific operational requirements.

## The Challenge of Unstructured Data: Why Generic Extraction Falls Short

The vast majority of the
 world's knowledge and data resides in unstructured formats like PDFs, emails, and scanned image

INFO:     [18:18:34] 📝 Report written for 'Schema-Based Document Extraction: Getting the Fields Your Business Actually Needs'


-guide-health-information
https://www.snowflake.com/en/developers/guides/doc-ai-invoice-reconciliation/
https://www.snowflake.com/en/engineering-blog/best-practices-snowflake
-document-ai/
https://www.index.dev/blog/ai-tools-for-database-schema-generation-optimization

📄 RESEARCH REPORT

# Schema-Based Document Extraction: Getting the Fields Your Business Actually Needs

In today's data-driven landscape, businesses are drowning in a sea of unstructured documents – from invoices and contracts to patient records and legal filings. The ability to efficiently extract precise, actionable data from these documents is no longer a luxury; it's an absolute necessity. Yet, many organizations still grapple with generic text extraction methods that yield an "Ambiguity Tax" of irrelevant information, high error rates, and endless manual cleanup. This is where **schema-based document extraction** emerges as a game-changer, allowing businesses to define exactly what data they need, ensuring outputs a

## 3. Multi-Retriever Research

Combine Gemini Grounding with other search engines for comprehensive results.

In [ ]:
async def multi_retriever_research():
    """
    Research using multiple retrievers for comprehensive results
    """
    # Configure multiple retrievers
    os.environ["RETRIEVER"] = "gemini_grounding,duckduckgo"
    
    print("🔍 Starting multi-retriever research...")
    print(f"Using retrievers: {os.environ['RETRIEVER']}\n")
    
    researcher = GPTResearcher(
        query="What are the latest breakthroughs in quantum computing?",
        report_type="research_report",
        verbose=True
    )
    
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    print("\n" + "="*80)
    print("📄 MULTI-RETRIEVER RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    sources = researcher.get_source_urls()
    print(f"\n📎 Found {len(sources)} sources across multiple retrievers")
    
    return report

# Run multi-retriever research
multi_report = await multi_retriever_research()

## 4. Fast Mode Research

Optimize for speed by disabling thinking and using faster models.

In [ ]:
async def fast_research():
    """
    Fast research with thinking disabled
    """
    import time
    
    # Configure for speed
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"  # Disable thinking
    
    print("⚡ Starting FAST research (thinking disabled)...\n")
    
    start_time = time.time()
    
    researcher = GPTResearcher(
        query="Quick summary of latest tech news this week",
        report_type="research_report",
        verbose=True
    )
    
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    elapsed_time = time.time() - start_time
    
    print("\n" + "="*80)
    print("📄 FAST RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    print(f"\n⏱️ Completed in: {elapsed_time:.2f} seconds")
    print(f"💰 Cost: ${researcher.get_costs():.4f}")
    
    return report, elapsed_time

# Run fast research
fast_report, duration = await fast_research()

## 5. Quality Mode Research

Enable thinking for higher quality analysis (slower but more thorough).

In [ ]:
async def quality_research():
    """
    High-quality research with thinking enabled
    """
    import time
    
    # Configure for quality
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ.pop("GEMINI_THINKING_BUDGET", None)  # Use default (thinking enabled)
    
    print("🎯 Starting QUALITY research (thinking enabled)...\n")
    
    start_time = time.time()
    
    researcher = GPTResearcher(
        query="What are the implications of recent AI safety research?",
        report_type="research_report",
        verbose=True
    )
    
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    elapsed_time = time.time() - start_time
    
    print("\n" + "="*80)
    print("📄 QUALITY RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    print(f"\n⏱️ Completed in: {elapsed_time:.2f} seconds")
    print(f"💰 Cost: ${researcher.get_costs():.4f}")
    
    return report, elapsed_time

# Run quality research
quality_report, quality_duration = await quality_research()

## 6. Detailed Research with Source Analysis

Examine the research sources and metadata in detail.

In [ ]:
async def detailed_research_with_sources():
    """
    Research with detailed source analysis
    """
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"
    
    print("🔬 Starting detailed research with source analysis...\n")
    
    researcher = GPTResearcher(
        query="What are the latest developments in climate technology?",
        report_type="research_report",
        verbose=True
    )
    
    # Conduct research
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    # Get detailed source information
    sources = researcher.get_research_sources()
    
    print("\n" + "="*80)
    print("📄 RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    # Detailed source analysis
    print("\n" + "="*80)
    print(f"📚 DETAILED SOURCE ANALYSIS ({len(sources)} sources)")
    print("="*80 + "\n")
    
    for i, source in enumerate(sources, 1):
        print(f"\n[Source {i}]")
        print(f"Title: {source.get('title', 'N/A')}")
        print(f"URL: {source.get('url', 'N/A')}")
        
        # Show snippet of content
        content = source.get('raw_content', '')
        if content:
            snippet = content[:200] + "..." if len(content) > 200 else content
            print(f"Content Preview: {snippet}")
        
        # Show images if available
        images = source.get('image_urls', [])
        if images:
            print(f"Images: {len(images)} found")
        
        print("-" * 80)
    
    # Get research context
    research_context = researcher.get_research_context()
    print(f"\n📊 Research Context Items: {len(research_context)}")
    
    return report, sources

# Run detailed research
detail_report, detail_sources = await detailed_research_with_sources()

## 7. Custom Query with Specific Configuration

Create a fully customized research task.

In [ ]:
async def custom_research():
    """
    Fully customized research configuration
    """
    # Reset to single retriever
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"
    
    print("⚙️ Starting custom configured research...\n")
    
    # Custom configuration
    researcher = GPTResearcher(
        query="YOUR CUSTOM QUERY HERE",
        report_type="research_report",  # Options: research_report, outline_report, etc.
        report_format="markdown",       # Options: markdown, apa, etc.
        tone="Objective",                # Tone of the report
        max_subtopics=5,                 # Max subtopics to explore
        verbose=True                     # Show detailed logs
    )
    
    # Conduct research
    context = await researcher.conduct_research()
    
    # Generate report
    report = await researcher.write_report()
    
    print("\n" + "="*80)
    print("📄 CUSTOM RESEARCH REPORT")
    print("="*80 + "\n")
    print(report)
    
    # Additional information
    print(f"\n📊 Statistics:")
    print(f"  - Sources: {len(researcher.get_source_urls())}")
    print(f"  - Context items: {len(researcher.get_research_context())}")
    print(f"  - Total cost: ${researcher.get_costs():.4f}")
    
    return report

# Uncomment to run custom research
# custom_report = await custom_research()

## 8. Comparison: Gemini Grounding vs Other Retrievers

Compare results from different search engines.

In [ ]:
async def compare_retrievers():
    """
    Compare research results from different retrievers
    """
    query = "What are the key features of Gemini 2.0?"
    
    results = {}
    
    # Test Gemini Grounding
    print("🔍 Testing Gemini Grounding...\n")
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"
    
    researcher1 = GPTResearcher(query=query, report_type="research_report", verbose=False)
    await researcher1.conduct_research()
    report1 = await researcher1.write_report()
    
    results["Gemini Grounding"] = {
        "report": report1[:500] + "...",
        "sources": len(researcher1.get_source_urls()),
        "cost": researcher1.get_costs()
    }
    
    # Test DuckDuckGo
    print("\n🦆 Testing DuckDuckGo...\n")
    os.environ["RETRIEVER"] = "duckduckgo"
    
    researcher2 = GPTResearcher(query=query, report_type="research_report", verbose=False)
    await researcher2.conduct_research()
    report2 = await researcher2.write_report()
    
    results["DuckDuckGo"] = {
        "report": report2[:500] + "...",
        "sources": len(researcher2.get_source_urls()),
        "cost": researcher2.get_costs()
    }
    
    # Display comparison
    print("\n" + "="*80)
    print("📊 RETRIEVER COMPARISON")
    print("="*80 + "\n")
    
    for retriever, data in results.items():
        print(f"\n{retriever}:")
        print(f"  Sources: {data['sources']}")
        print(f"  Cost: ${data['cost']:.4f}")
        print(f"  Report Preview: {data['report'][:150]}...")
        print("-" * 80)
    
    return results

# Run comparison
comparison_results = await compare_retrievers()

## 9. Export Reports

Save your research reports to files.

In [ ]:
async def export_research():
    """
    Conduct research and export to file
    """
    os.environ["RETRIEVER"] = "gemini_grounding"
    os.environ["GEMINI_THINKING_BUDGET"] = "0"
    
    print("📝 Conducting research for export...\n")
    
    researcher = GPTResearcher(
        query="Summary of recent breakthroughs in renewable energy",
        report_type="research_report",
        verbose=True
    )
    
    context = await researcher.conduct_research()
    report = await researcher.write_report()
    
    # Export to markdown file
    output_file = "research_report.md"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("# Research Report\n\n")
        f.write(report)
        f.write("\n\n## Sources\n\n")
        for i, source in enumerate(researcher.get_source_urls(), 1):
            f.write(f"{i}. {source}\n")
    
    print(f"\n✅ Report exported to: {output_file}")
    print(f"📊 Report length: {len(report)} characters")
    print(f"💰 Total cost: ${researcher.get_costs():.4f}")
    
    return output_file

# Export research
output_file = await export_research()
print(f"\n📄 You can now read the report in: {output_file}")

## 10. Tips and Best Practices

### Configuration Tips

1. **For Speed**: Set `GEMINI_THINKING_BUDGET=0`
2. **For Quality**: Leave thinking budget unset (default)
3. **For Current Events**: Use `gemini_grounding` retriever
4. **For Comprehensive Research**: Use multiple retrievers
5. **For JavaScript Sites**: Use `SCRAPER=browser`

### Cost Optimization

- Disable thinking for production systems
- Use flash-lite model for simple queries
- Limit max_subtopics for focused research
- Monitor costs with `researcher.get_costs()`

### Troubleshooting

If you encounter issues:
1. Check your API key is set correctly
2. Verify `google-genai` package is installed
3. Enable verbose mode: `verbose=True`
4. Check the logs for specific error messages

### Additional Resources

- Documentation: https://docs.gptr.dev
- Gemini Setup Guide: See `GEMINI_SETUP.md` in repo root
- Gemini API Docs: https://ai.google.dev/gemini-api/docs

## Summary

This notebook demonstrated:
- ✅ Basic research with Gemini Grounding
- ✅ Multi-retriever research
- ✅ Fast vs Quality mode
- ✅ Detailed source analysis
- ✅ Custom configurations
- ✅ Retriever comparisons
- ✅ Report exporting

You're now ready to use GPT Researcher with Gemini 2.5! 🚀